# Frequency Estimation: NLS vs DFT

The pipeline notebooks ([`0.0`](0.0_quick-start.ipynb)–[`0.3`](0.3_profile-likelihood-q.ipynb))
call `RingDownAnalyzer`, which hides the frequency estimators behind a single
`analyze_array()` call. This notebook opens that box: it uses
`NLSFrequencyEstimator` and `DFTFrequencyEstimator` directly on one synthetic
record, so you can see what each method measures, how accurate it is, and what it
costs.

The model is a single-mode ring-down,

$$
x(t) = A_0 e^{-t/\tau}\cos(2\pi f_0 t + \phi_0) + c + \epsilon(t),
$$

with quality factor $Q = \pi f_0 \tau$. Both estimators target $f_0$; both can also
return $\tau$ and $Q$, but by different routes:

- **NLS** fits the full model in the time domain and solves for $f$, $\tau$, $A_0$,
  $\phi_0$, and $c$ jointly. It is the statistically efficient choice.
- **DFT** finds the spectral peak and refines it with a Lorentzian fit of the
  surrounding bins — a ring-down peak is Lorentzian, with half-width set by
  $1/(\pi\tau)$. It is cheap and needs no starting guess, which is exactly what
  makes it a good initializer for NLS.

**Prerequisites:** [`0.0_quick-start.ipynb`](0.0_quick-start.ipynb) for the signal
model and the analyzer output fields. No external data is needed here.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal.windows import kaiser

from ringdownanalysis import (
    CRLBCalculator,
    DFTFrequencyEstimator,
    NLSFrequencyEstimator,
    RingDownSignal,
    plots,
)

plots.apply_plotting_style()

## 1. One coherent record

Estimator accuracy on a ring-down is governed by the record length measured in decay
times, $T/\tau$: past a few decay times the signal has died into the noise and extra
samples add nothing. The technical note and `examples/usage_example.py` use
$f_0 = 5$ Hz, $f_s = 100$ Hz, $N = 10^6$, $Q = 10^4$, which is $T/\tau \approx 15.7$.

Here we keep that same $T/\tau$ with ten times fewer samples ($N = 10^5$,
$Q = 10^3$), so every fit below runs in well under a second.

In [ ]:
f0 = 5.0  # Hz
fs = 100.0  # Hz
N = 100_000  # samples
A0 = 1.0
snr_db = 60.0  # initial SNR
Q_true = 1000.0

signal = RingDownSignal(f0=f0, fs=fs, N=N, A0=A0, snr_db=snr_db, Q=Q_true)
rng = np.random.default_rng(42)
t, x, phi0 = signal.generate(rng=rng)

tau_true = signal.tau
print(f"T = {signal.T:.1f} s, tau = {tau_true:.3f} s, T/tau = {signal.T / tau_true:.2f}")
print(f"noise sigma = {signal.sigma:.3e} (initial SNR {snr_db:.0f} dB)")
print(f"true f0 = {f0} Hz, true Q = {Q_true:.1f}")

## 2. Nonlinear least squares

`NLSFrequencyEstimator` has two entry points:

- `estimate(x, fs)` returns the frequency only.
- `estimate_full(x, fs)` returns an `EstimationResult` with `f`, `tau`, `Q`, plus the
  convergence flags (`success`, `used_fallback`, `message`, `nfev`).

Always read the flags. When a fit fails or fails an internal sanity check, the
estimator does not raise: it returns the heuristic initializer with
`success=False` and `used_fallback=True`. Treating that as a measurement is the most
common way to get a confidently wrong Q.

`tau_known=None` asks for a joint fit of frequency and decay time. Passing a known
`tau_known` fixes the decay and fits frequency only, which is useful when $\tau$ has
already been established from a longer record.

In [ ]:
nls = NLSFrequencyEstimator(tau_known=None)

start = time.perf_counter()
f_nls = nls.estimate(x, fs)
t_nls_freq = time.perf_counter() - start
print(f"estimate():      f = {f_nls:.9f} Hz  ({t_nls_freq * 1e3:.1f} ms)")

start = time.perf_counter()
res_nls = nls.estimate_full(x, fs)
t_nls = time.perf_counter() - start

print(f"estimate_full(): f = {res_nls.f:.9f} Hz  (error {res_nls.f - f0:+.3e} Hz)")
print(f"                 tau = {res_nls.tau:.4f} s  (error {res_nls.tau - tau_true:+.3e} s)")
print(f"                 Q = {res_nls.Q:.4f}  (error {res_nls.Q - Q_true:+.3e})")
print(f"                 success={res_nls.success}, fallback={res_nls.used_fallback}, nfev={res_nls.nfev}")
print(f"                 message: {res_nls.message}")
print(f"                 runtime: {t_nls * 1e3:.1f} ms")
print(f"fitted decay reaches 1/e after {res_nls.tau * f0:.0f} carrier cycles")

## 3. DFT peak with Lorentzian refinement

`DFTFrequencyEstimator` zero-pads the record (`pad_factor=4` by default), takes the
FFT, picks the largest bin above `f_min`, and fits a Lorentzian to the bins around it
to interpolate between the grid points.

Two arguments matter most:

- **`window`** — `"rect"` (no window), `"hann"`, `"kaiser"`, or `"blackman"`.
  Leaving the record untapered keeps the peak shape exactly Lorentzian, which is what
  the refinement step fits, at the price of sinc-like leakage skirts. A taper
  suppresses the skirts but multiplies the decaying envelope by the window: the peak is
  no longer the assumed Lorentzian, and the early, high-amplitude samples that carry
  most of the frequency information get down-weighted. On an isolated ring-down that
  trade is a net loss — an order of magnitude in the table below — so `"rect"` is the
  accurate choice. Tapers pay off only when a strong neighbouring line would otherwise
  leak into the peak.
- **`f_min`** — bins below `f_min` are excluded from the peak search. Phase records
  with baseline wander have a large low-frequency component, and without `f_min` the
  peak search can lock onto the wander instead of the resonance. This is essential on
  real data (see [`0.5_odin-phasemeter-data.ipynb`](0.5_odin-phasemeter-data.ipynb));
  the default `f_min=0.0` is fine for this clean synthetic.

Note what `estimate_full()` does here: the DFT supplies the frequency, then `tau` is
obtained from an NLS fit with the frequency held fixed. So a DFT `Q` inherits the DFT
frequency error but is otherwise a time-domain decay fit — and `estimate_full()` costs
about as much as a full NLS fit, while `estimate()` is the genuinely cheap call. The
table reports both.

In [ ]:
dft_rows = []
dft_results = {}

for window in ("rect", "hann", "kaiser"):
    dft = DFTFrequencyEstimator(window=window)

    start = time.perf_counter()
    dft.estimate(x, fs)
    t_freq = time.perf_counter() - start

    start = time.perf_counter()
    res = dft.estimate_full(x, fs)
    t_full = time.perf_counter() - start

    dft_results[window] = res
    dft_rows.append(
        {
            "window": window,
            "f (Hz)": res.f,
            "f error (Hz)": res.f - f0,
            "tau error (s)": res.tau - tau_true,
            "Q error": res.Q - Q_true,
            "estimate() ms": t_freq * 1e3,
            "estimate_full() ms": t_full * 1e3,
        }
    )

pd.DataFrame(dft_rows).set_index("window")

## 4. Side by side

The natural yardstick is the Cramér–Rao lower bound: no unbiased estimator can beat
it, so `|error| / CRLB std` says how much accuracy a method leaves on the table. On a
single record the ratio fluctuates — one draw of the noise can land closer to the
truth than the bound by luck — so read the numbers below as an order of magnitude, not
a measurement of efficiency. The ensemble version of this comparison is
[`0.6_monte-carlo-crlb.ipynb`](0.6_monte-carlo-crlb.ipynb).

The Q errors come out nearly identical for every method, which is expected: the DFT
only sets the frequency, and $\tau$ always comes from a time-domain fit. Frequency is
where the methods actually differ.

The left panel of the figure shows the window trade-off directly. The untapered
spectrum is a clean Lorentzian core sitting on leakage skirts; the Kaiser window
removes the skirts but reshapes the core, which is what costs the fit its accuracy.

In [ ]:
crlb = CRLBCalculator()
crlb_std_f = crlb.standard_deviation(A0, signal.sigma, fs, N, tau_true)
crlb_std_q = crlb.q_standard_deviation(A0, signal.sigma, fs, N, tau_true, f0)

print(f"CRLB std(f) = {crlb_std_f:.3e} Hz")
print(f"CRLB std(Q) = {crlb_std_q:.3e}")

rows = [
    {
        "method": "NLS",
        "f error (Hz)": res_nls.f - f0,
        "|f error| / CRLB std": abs(res_nls.f - f0) / crlb_std_f,
        "Q error": res_nls.Q - Q_true,
        "|Q error| / CRLB std": abs(res_nls.Q - Q_true) / crlb_std_q,
        "frequency runtime (ms)": t_nls_freq * 1e3,
    }
]
for window, res in dft_results.items():
    rows.append(
        {
            "method": f"DFT ({window})",
            "f error (Hz)": res.f - f0,
            "|f error| / CRLB std": abs(res.f - f0) / crlb_std_f,
            "Q error": res.Q - Q_true,
            "|Q error| / CRLB std": abs(res.Q - Q_true) / crlb_std_q,
            "frequency runtime (ms)": [
                r["estimate() ms"] for r in dft_rows if r["window"] == window
            ][0],
        }
    )

comparison = pd.DataFrame(rows).set_index("method")
comparison

In [ ]:
# Where the estimates sit on the spectrum (same zero-padded FFT the estimator uses)
n_pad = 4 * N
freqs = np.fft.rfftfreq(n_pad, d=1.0 / fs)
windows = {"rect": np.ones(N), "kaiser": kaiser(N, 9.0)}
spectra = {}
for window, w in windows.items():
    xw = np.zeros(n_pad)
    xw[:N] = (x - x.mean()) * w
    spectra[window] = np.abs(np.fft.rfft(xw)) ** 2

band = (freqs > f0 - 0.02) & (freqs < f0 + 0.02)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
for window, power in spectra.items():
    ax.semilogy(freqs[band], power[band] / power[band].max(), label=f"{window} window")
ax.axvline(f0, color="C2", linestyle="--", linewidth=1.2, label=f"true f0 = {f0} Hz")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Normalized power")
ax.set_title("Ring-down peak: Lorentzian core vs leakage skirts")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
labels = list(comparison.index)
errors = np.abs(comparison["f error (Hz)"].to_numpy())
ax.bar(labels, errors, color=["C0"] + ["C1"] * (len(labels) - 1), alpha=0.85)
ax.axhline(crlb_std_f, color="C2", linestyle="--", linewidth=1.2, label="CRLB std")
ax.set_yscale("log")
ax.set_ylabel("|frequency error| (Hz)")
ax.set_title("Single-record frequency error")
ax.tick_params(axis="x", rotation=20)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Record length and the choice of method

The reason to care about $T/\tau$ is that a ring-down stops carrying information once
it has decayed. While $T \lesssim \tau$ the accuracy improves steeply with record
length — the bound scales as $T^{-3/2}$, the same as for a constant-amplitude tone —
but past a few decay times the added samples are pure noise and the bound flattens out
at a value set by $\tau$ alone. The CRLB column below shows both regimes: it improves
between $T/\tau = 0.5$ and $T/\tau \approx 5$, then stops moving.

The loop repeats both estimators on each length, averaging $|f - f_0|$ over five noise
realizations so the numbers are not dominated by a single draw. Five realizations is
enough to see the trend and far too few to quote an efficiency — that is what
[`0.6_monte-carlo-crlb.ipynb`](0.6_monte-carlo-crlb.ipynb) is for.

In [ ]:
dft_rect = DFTFrequencyEstimator(window="rect")
length_rows = []

for n_samples in (3_000, 10_000, 30_000, 100_000):
    sig_n = RingDownSignal(f0=f0, fs=fs, N=n_samples, A0=A0, snr_db=snr_db, Q=Q_true)
    err_nls, err_dft = [], []
    for seed in range(5):
        _, x_n, _ = sig_n.generate(rng=np.random.default_rng(1000 + seed))
        err_nls.append(abs(nls.estimate(x_n, fs) - f0))
        err_dft.append(abs(dft_rect.estimate(x_n, fs) - f0))
    length_rows.append(
        {
            "N": n_samples,
            "T/tau": sig_n.T / sig_n.tau,
            "mean |f error| NLS (Hz)": float(np.mean(err_nls)),
            "mean |f error| DFT (Hz)": float(np.mean(err_dft)),
            "CRLB std (Hz)": crlb.standard_deviation(A0, sig_n.sigma, fs, n_samples, sig_n.tau),
        }
    )

pd.DataFrame(length_rows).set_index("N")

## 6. Choosing between them, and where they reappear

**Use NLS** when you want the number you will quote. It is efficient, it returns
$\tau$ and $Q$ from the same fit, and its cost is modest because the library seeds it
from a DFT-based initializer — the fit above converged in a handful of function
evaluations.

**Use DFT** when you need a frequency without a starting guess and cheaply: a first
look at an unfamiliar record, a band-limited peak search on a drifting record via
`f_min`, or a seed for something else. Use `window="rect"` unless leakage from a
neighbouring line forces a taper.

**Neither is a substitute for Q inference.** Both report $Q = \pi f \tau$ from a
point-estimate $\tau$, which says nothing about whether the record actually identified
$\tau$. A short or high-Q record can return a large finite $\tau$ that the data do not
support — that is what [`0.3_profile-likelihood-q.ipynb`](0.3_profile-likelihood-q.ipynb)
diagnoses, and why the pipeline gates raw fits behind status fields.

In `RingDownAnalyzer.analyze_array()` results, these estimators appear as:

| Field | Source |
| --- | --- |
| `f_nls` | `NLSFrequencyEstimator.estimate_full().f` on the selected crop |
| `f_dft` | `DFTFrequencyEstimator.estimate_full().f` on the same crop |
| `Q_nls_raw`, `Q_dft_raw` | raw `EstimationResult.Q` from each estimator |
| `Q_nls`, `Q_dft` | the same values, populated only when the estimate passes validation |

Both are phase-coherent fits, so both are biased by frequency drift on real records.
The pipeline checks `coherence_ratio` before trusting them and prefers the
drift-immune `Q_demod` when it fires — see README § "Which Q should I trust?".

### Next steps

- Real measurement files, drift, and `Q_selected`: [`0.1_drifting-resonators.ipynb`](0.1_drifting-resonators.ipynb)
- Zipped Moku phasemeter exports: [`0.5_odin-phasemeter-data.ipynb`](0.5_odin-phasemeter-data.ipynb)
- CRLB and Monte Carlo foundations: [`0.6_monte-carlo-crlb.ipynb`](0.6_monte-carlo-crlb.ipynb)